

```
# Ce texte est au format code
```

****

# Phase 3 — Modèles ML : SVM (Support vector machine)
**Auteur : Ihssane Moutchou | PFA 4ème Année**

In [ ]:
import os

data_path = os.path.join(REPO_PATH, 'data')

X_train = joblib.load(os.path.join(data_path, 'X_train.pkl'))
X_test  = joblib.load(os.path.join(data_path, 'X_test.pkl'))
y_train = pd.read_csv(os.path.join(data_path, 'y_train.csv')).squeeze() # .squeeze() to convert DataFrame to Series if it's a single column
y_test  = pd.read_csv(os.path.join(data_path, 'y_test.csv')).squeeze() # .squeeze() to convert DataFrame to Series if it's a single column

print(f"Train : {X_train.shape[0]:,} avis \u00d7 {X_train.shape[1]:,} features")
print(f"Test  : {X_test.shape[0]:,} avis")
print(f"\nClasses : {sorted(y_train.unique())}")
print(f"\nDistribution train :")
print(pd.Series(y_train).value_counts())

In [ ]:
def evaluer_modele(nom, modele, X_train, X_test, y_train, y_test):
    """
    Entraîne le modèle, affiche toutes les métriques, retourne un dict de résultats.
    """
    # Entraînement
    modele.fit(X_train, y_train)

    # Prédictions
    y_pred = modele.predict(X_test)

    # Métriques principales
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    f1_m = f1_score(y_test, y_pred, average='macro')

    print(f"\n{'='*55}")
    print(f"  MODÈLE : {nom}")
    print(f"{'='*55}")
    print(f"  Accuracy         : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  F1 weighted      : {f1:.4f}")
    print(f"  F1 macro         : {f1_m:.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")

    return {
        'nom': nom,
        'modele': modele,
        'y_pred': y_pred,
        'accuracy': acc,
        'f1_weighted': f1,
        'f1_macro': f1_m
    }

print("Fonction evaluer_modele() prête")

**LinearSVC avec GridSearch**

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV

# LinearSVC ne donne pas de probabilités par défaut.
# CalibratedClassifierCV corrige ça — nécessaire pour l'API FastAPI
# qui renverra des scores de confiance.

svc_base = CalibratedClassifierCV(
    LinearSVC(random_state=42, max_iter=2000),
    cv=3
)

# C contrôle la marge : grand C = marges plus dures (plus overfitting possible)
param_grid_svc = {
    'estimator__C': [0.01, 0.1, 1, 10, 100]
}

gs_svc = GridSearchCV(
    svc_base,
    param_grid_svc,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

print("GridSearch LinearSVC en cours (peut prendre 3-5 min)...")
gs_svc.fit(X_train, y_train)

print(f"Meilleur C : {gs_svc.best_params_}")
print(f"F1 CV      : {gs_svc.best_score_:.4f}")

resultats_svc = evaluer_modele(
    "LinearSVC (C=" + str(gs_svc.best_params_['estimator__C']) + ")",
    gs_svc.best_estimator_,
    X_train, X_test, y_train, y_test
)

**Analyse par classe (point fort du SVM)**

In [ ]:
# Quelles erreurs fait le SVM ?
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np

report = classification_report(
    y_test, resultats_svc['y_pred'],
    output_dict=True
)
df_report = pd.DataFrame(report).T.iloc[:3]  # 3 classes seulement

print("Analyse par classe :")
print(df_report[['precision','recall','f1-score','support']].round(3))

# Graphique precision vs recall par classe
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
width = 0.3
bars1 = ax.bar(x - width, df_report['precision'], width, label='Precision', color='steelblue')
bars2 = ax.bar(x,         df_report['recall'],    width, label='Recall',    color='coral')
bars3 = ax.bar(x + width, df_report['f1-score'],  width, label='F1',        color='mediumseagreen')

ax.set_xticks(x)
ax.set_xticklabels(['NEGATIF', 'NEUTRE', 'POSITIF'])
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title('LinearSVC — Precision / Recall / F1 par classe')
plt.tight_layout()
plt.show()

In [ ]:
import joblib
import os

# Define the base path for models using REPO_PATH
models_path = os.path.join(REPO_PATH, 'models')

# Ensure the models directory exists
os.makedirs(models_path, exist_ok=True)

# Save the best estimator
joblib.dump(gs_svc.best_estimator_, os.path.join(models_path, 'linear_svc.pkl'))
print("linear_svc.pkl sauvegardé")

# Save the results dictionary
joblib.dump({
    'linear_svc': resultats_svc
}, os.path.join(models_path, 'resultats_ihssane.pkl'))

print("Phase 3 Ihssane — TERMINÉE ✓")
print(f"SVM F1 = {resultats_svc['f1_weighted']:.4f}")